In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score

In [3]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

print(df.shape)
print(df.head())
print(df.dtypes)

(5000, 9)
      lead_source       industry employment_status       location  \
0  organic_search     technology          employed         europe   
1    social_media     technology          employed  south_america   
2        referral         retail          employed         europe   
3        paid_ads  manufacturing           student            NaN   
4        referral  manufacturing           student  north_america   

   annual_income  number_of_courses_viewed  interaction_count  lead_score  \
0        88160.0                         3                  4        0.64   
1        72688.0                         3                  7        0.72   
2        44697.0                         3                  4        0.58   
3            NaN                         3                  6        0.57   
4        24062.0                         4                  5        0.62   

   converted  
0          1  
1          1  
2          1  
3          0  
4          1  
lead_source           

Data preparation
Check if the missing values are presented in the features.
If there are missing values:
For categorical features, replace them with 'NA'
For numerical features, replace with with 0.0

In [4]:
#Check for missing values:
print(df.isnull().sum())

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


If there are missing values:
For categorical features, replace them with 'NA'
For numerical features, replace with with 0.0

In [5]:
categorical_columns = df.select_dtypes(
    include=["object", "str", "string"]
).columns

numerical_columns = df.select_dtypes(
    exclude=["object", "str", "string"]
).columns

df[categorical_columns] = df[categorical_columns].fillna("NA")
df[numerical_columns] = df[numerical_columns].fillna(0.0)

In [6]:
print(df.isnull().sum())

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


Question 1
What is the most frequent observation (mode) for the column industry?

NA
technology
healthcare
retail

In [7]:
print(df["industry"].mode())

0    technology
Name: industry, dtype: str


In [8]:
print(df["industry"].value_counts())

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

interaction_count and lead_score
number_of_courses_viewed and lead_score
number_of_courses_viewed and interaction_count
annual_income and interaction_count
Only consider the pairs above when answering this question.

In [9]:
numerical_df = df.select_dtypes(include="number").drop(
    columns=["converted"]
)

corr = numerical_df.corr()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count")
]

for col1, col2 in pairs:
    print(
        f"{col1} and {col2}: "
        f"{corr.loc[col1, col2]:.4f}"
    )

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


Split the data


In [11]:
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train["converted"].values
y_val = df_val["converted"].values

df_train = df_train.drop(columns=["converted"])
df_val = df_val.drop(columns=["converted"])

print("Training features:", df_train.shape)
print("Validation features:", df_val.shape)
print("Test dataset:", df_test.shape)
print("Training target:", y_train.shape)
print("Validation target:", y_val.shape)

Training features: (3000, 8)
Validation features: (1000, 8)
Test dataset: (1000, 9)
Training target: (3000,)
Validation target: (1000,)


Question 3
Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score?

industry

location

lead_source

employment_status

In [12]:
categorical_columns = df_train.select_dtypes(
    include=["object", "str", "string"]
).columns

def calculate_mi(series):
    return mutual_info_score(series, y_train)

mi_scores = df_train[categorical_columns].apply(calculate_mi)

print(mi_scores.sort_values(ascending=False).round(2))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


Question 4
Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.
To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get?

0.55

0.65

0.75

0.85

In [13]:
# Convert categorical features into numerical features
dv = DictVectorizer(sparse=False)

train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# Train Logistic Regression
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

# Predict validation results
y_pred = model.predict(X_val)

# Calculate accuracy
accuracy = accuracy_score(y_val, y_pred)

print("Accuracy:", accuracy)
print("Rounded accuracy:", round(accuracy, 2))

Accuracy: 0.645
Rounded accuracy: 0.65


Question 5

Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?

'lead_source'

'number_of_courses_viewed'

'interaction_count'

In [14]:
features = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]

# Original accuracy without rounding
original_accuracy = accuracy_score(
    y_val, model.predict(X_val)
)

print("Original accuracy:", original_accuracy)

for feature in features:

    # Remove one feature from training and validation data
    train_reduced = df_train.drop(columns=[feature])
    val_reduced = df_val.drop(columns=[feature])

    # One-hot encoding
    dv_reduced = DictVectorizer(sparse=False)

    X_train_reduced = dv_reduced.fit_transform(
        train_reduced.to_dict(orient="records")
    )

    X_val_reduced = dv_reduced.transform(
        val_reduced.to_dict(orient="records")
    )

    # Train the model using the same parameters as Q4
    model_reduced = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model_reduced.fit(X_train_reduced, y_train)

    # Calculate accuracy without the feature
    accuracy_reduced = accuracy_score(
        y_val, model_reduced.predict(X_val_reduced)
    )

    # Calculate the difference
    difference = original_accuracy - accuracy_reduced

    print(f"\nFeature removed: {feature}")
    print("Accuracy:", accuracy_reduced)
    print("Difference:", difference)

Original accuracy: 0.645

Feature removed: lead_source
Accuracy: 0.642
Difference: 0.0030000000000000027

Feature removed: number_of_courses_viewed
Accuracy: 0.643
Difference: 0.0020000000000000018

Feature removed: interaction_count
Accuracy: 0.601
Difference: 0.04400000000000004


Question 6

Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

0.000001

0.00001

0.0001

0.001

In [15]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

results = []

for C in C_values:
    model_reg = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42
    )

    model_reg.fit(X_train, y_train)

    y_pred = model_reg.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)

    results.append((C, accuracy))

    print(f"C = {C}, Accuracy = {accuracy:.3f}")

C = 1e-06, Accuracy = 0.598
C = 1e-05, Accuracy = 0.598
C = 0.0001, Accuracy = 0.613
C = 0.001, Accuracy = 0.645
